In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, min, max, when, from_unixtime,
    datediff, to_date
)

spark = (
    SparkSession.builder
    .appName("03_Preprocess_Feature_Engineering")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"
HDFS_SILVER = "hdfs://namenode:8020/netflix-recsys/silver"
HDFS_GOLD = "hdfs://namenode:8020/netflix-recsys/gold"

ratings = spark.read.parquet(f"{HDFS_RAW}/ratings")
movies = spark.read.parquet(f"{HDFS_RAW}/movies")

In [ ]:
ratings_cleaned = (
    ratings
    .select(
        col("userId").cast("int"),
        col("movieId").cast("int"),
        col("rating").cast("float"),
        col("timestamp").cast("long")
    )
    .dropna()
    .filter((col("rating") >= 0.5) & (col("rating") <= 5.0))
)

movies_cleaned = (
    movies
    .select(
        col("movieId").cast("int"),
        col("title").cast("string"),
        col("genres").cast("string")
    )
    .dropna(subset=["movieId", "title"])
)

In [ ]:
user_profile = (
    ratings_cleaned
    .withColumn("rating_date", to_date(from_unixtime(col("timestamp"))))
    .groupBy("userId")
    .agg(
        count("*").alias("num_ratings"),
        min("rating_date").alias("first_rating_date"),
        max("rating_date").alias("last_rating_date")
    )
    .withColumn(
        "active_days",
        datediff(col("last_rating_date"), col("first_rating_date"))
    )
)

long_term_users = (
    user_profile
    .filter((col("num_ratings") >= 50) & (col("active_days") >= 180))
    .select("userId")
)

long_term_ratings = ratings_cleaned.join(long_term_users, "userId", "inner")

print("Long-term users:", long_term_users.count())
print("Long-term ratings:", long_term_ratings.count())

In [ ]:
labeled_interactions = (
    long_term_ratings
    .withColumn("label", when(col("rating") >= 4.0, 1).otherwise(0))
)

In [ ]:
als_train, als_test = long_term_ratings.randomSplit([0.8, 0.2], seed=42)

In [ ]:
ratings_cleaned.write.mode("overwrite").parquet(f"{HDFS_SILVER}/ratings_cleaned")
movies_cleaned.write.mode("overwrite").parquet(f"{HDFS_SILVER}/movies_cleaned")

user_profile.write.mode("overwrite").parquet(f"{HDFS_GOLD}/user_profile")
long_term_users.write.mode("overwrite").parquet(f"{HDFS_GOLD}/long_term_users")
long_term_ratings.write.mode("overwrite").parquet(f"{HDFS_GOLD}/long_term_ratings")
labeled_interactions.write.mode("overwrite").parquet(f"{HDFS_GOLD}/labeled_interactions")

als_train.write.mode("overwrite").parquet(f"{HDFS_GOLD}/als_train")
als_test.write.mode("overwrite").parquet(f"{HDFS_GOLD}/als_test")